# BAB 1 — Business Understanding
## Klasifikasi penggunaan/penutupan lahan Jawa Timur dari Sentinel-2

## 1.1 Latar belakang

**Land use vs land cover.** *Land cover* = tutupan fisik permukaan bumi yang dapat diamati (mis. air, vegetasi, bangunan). *Land use* = fungsi/pemanfaatan oleh manusia (mis. sawah sebagai lahan produksi pangan, permukiman sebagai tempat tinggal). Keduanya berhubungan tetapi tidak identik: satu tutupan fisik bisa punya fungsi berbeda, dan satu fungsi bisa menempati tutupan berbeda.

**Posisi 6 kelas project di antara keduanya:**

- Lebih dekat ke *land cover*: Laut, Danau (tubuh air sebagai tutupan fisik).
- Lebih dekat ke *land use* (tutupan yang didefinisikan oleh pemanfaatan/budidaya): Sawah, Bangunan / Permukiman.
- Campuran keduanya: Mangrove (tutupan vegetasi pesisir yang juga menjadi objek pengelolaan pesisir) dan Lahan hijau (kelas operasional agregat berisi banyak tutupan vegetasi budidaya maupun semi-alami — lihat 1.5; kecenderungan mencampur ini dibahas sebagai keterbatasan, bukan keunggulan).

**Mengapa informasi spasial.** Peta menunjukkan *di mana* setiap kelas berada sehingga luas per kelas dapat dihitung dari hasil klasifikasi dan dapat dibandingkan antar wilayah. Tanpa dimensi spasial, statistik tabular tidak dapat dipakai untuk tata ruang.

**Hubungan dengan tata ruang dan kebijakan.** Informasi penggunaan/penutupan lahan cenderung dipakai sebagai salah satu bahan pertimbangan penataan ruang dan kebijakan sektoral (mis. perlindungan lahan pertanian, pengendalian permukiman, pengelolaan pesisir/mangrove dan perairan).peta project BUKAN produk rencana tata ruang resmi dan TIDAK dapat dipakai untuk klaim kausal kebijakan.

**Mengapa Jawa Timur.** Jawa Timur mencakup daratan dengan variasi tinggi (kawasan urban, pertanian intensif, hutan/perkebunan, pesisir dan pulau) plus wilayah laut di sekitarnya, sehingga keenam kelas project semuanya relevan dalam satu area studi. Batas area studi dibuktikan dari isi data pada Lampiran A3 (bukan dari nama file).

**Mengapa Sentinel-2 (spesifikasi umum sensor — bukan data project).** Sentinel-2 (misi Copernicus/ESA, akses data terbuka) membawa sensor MSI multispektral dengan kanal tampak, near-infrared (NIR) dan shortwave-infrared (SWIR); konstelasi dua satelit cenderung memberi revisit sekitar 5 hari di ekuator. Kombinasi ini cenderung membantu membedakan vegetasi, air, dan lahan terbangun: NIR cenderung sensitif terhadap vegetasi hidup, SWIR cenderung sensitif terhadap kadar air/struktur permukaan, sedangkan kanal tampak memberi kecerahan dasar. Project ini memakai enam band sesuai metadata (B02, B03, B04, B08, B11, B12 — diverifikasi dari isi `s2_jatim_metadata.json` pada Lampiran A2). Sumber spesifikasi umum: ESA Sentinel-2 MSI overview (https://scihub.copernicus.eu / https://sentinels.copernicus.eu).

## 1.2 Tujuan analisis

Membangun sistem klasifikasi penggunaan/penutupan lahan Jawa Timur dari citra Sentinel-2 dan machine learning, dengan keluaran:

1. Model terbaik yang dipilih berdasarkan **CV (GroupKFold) pada data training** (group = polygon/FID), bukan dari test set.
2. Peta klasifikasi seluruh raster (`klasifikasi_jawa_timur.tif`) dari model terbaik.
3. Evaluasi jujur satu kali pada test set di akhir (Bab 5), termasuk luas per kelas.
4. Hasil ditampilkan di **WebGIS di dalam Jupyter notebook** (karena web statis project = Jupyter Book), dengan **Streamlit (`app.py`) sebagai pelengkap**.

Label training berasal dari sumber sekunder yang dibentuk sendiri (bukan survei lapangan) — didokumentasikan jujur di Bab 2. Maka metrik mengukur **kesesuaian dengan label sumber ini**, bukan akurasi terhadap kondisi lapangan.

## 1.3 Pertanyaan analisis

| # | Pertanyaan | Dijawab di | Cara menjawab |
|---|---|---|---|
| Q1 | Bagaimana karakteristik citra Sentinel-2 project? | Bab 2 | Baca `s2_jatim.tif` + `s2_jatim_metadata.json` dari data (bukan asumsi) |
| Q2 | Bagaimana karakteristik training polygon? | Bab 2 | Saring validitas V1–V6, catat per alasan; pilih maks 50 valid per kelas |
| Q3 | Bagaimana polygon menjadi data ML? | Bab 3 | Ekstraksi pixel per polygon (aturan centroid), capping 500, polygon-mean; split per polygon (FID) |
| Q4 | Apakah pixel-based atau polygon-mean menghasilkan model lebih baik? | Bab 4–5 | Bandingkan lewat CV pada training; jawab hanya dari hasil, tanpa bias awal |
| Q5 | Model mana performanya terbaik? | Bab 4–5 | Pilih via CV training (lapor std antar fold + ketidakpastian); test dipakai sekali di akhir |
| Q6 | Bagaimana model dipakai membuat peta Jatim? | Bab 5–6 | Klasifikasi seluruh raster per window/blok → `klasifikasi_jawa_timur.tif` → WebGIS |

Q4 dan Q5 **belum dijawab** di notebook ini — menjawabnya sekarang berarti mengarang.

## 1.4 Definisi kelas operasional

Enam kelas berikut adalah **keputusan eksperimen (boleh hardcode nama/path)**. Tidak ada kelas ketujuh. Kecocokannya dengan isi file diverifikasi pada sel kode sesudah tabel (DATA AKTUAL PROJECT).

| class_id | class_name | Catatan operasional |
|---|---|---|
| 1 | Sawah | Lahan budidaya basah (sawah/bera/panen ikut dinamika snapshot citra) |
| 2 | Bangunan / Permukiman | Lahan terbangun permukiman dan campurannya |
| 3 | Mangrove | Vegetasi pesisir/payau |
| 4 | Lahan hijau | KELAS AGREGAT: kebun, tegalan, hutan, semak, padang rumput, dsb. (ambigu — lihat 1.5) |
| 5 | Laut | Perairan laut dalam area studi |
| 6 | Danau | Danau alami + waduk/embung buatan (kolom `source_class` dirinci di Bab 2) |

In [1]:
# DATA AKTUAL PROJECT — verifikasi pemetaan class_id -> class_name dari isi file training.
# Bukan hardcode: unik dihitung dari file. (Distribusi jumlah per kelas dikerjakan di Bab 2.)
import geopandas as gpd

TRAIN_GEOJSON = "../data/training_samples_jawa_timur.geojson"
EXPECTED = {1: "Sawah", 2: "Bangunan / Permukiman", 3: "Mangrove",
            4: "Lahan hijau", 5: "Laut", 6: "Danau"}

cols = ["class_id", "class_name"]
gdf_classes = gpd.read_file(TRAIN_GEOJSON, ignore_geometry=True)[cols]
pairs = gdf_classes.drop_duplicates().sort_values("class_id").reset_index(drop=True)
print(f"DATA AKTUAL PROJECT: pasangan unik class_id/class_name di file = {len(pairs)}")
for _, r in pairs.iterrows():
    cocok = "COCOK" if EXPECTED.get(int(r['class_id'])) == r['class_name'] else "TIDAK COCOK"
    print(f"  class_id={int(r['class_id'])} class_name='{r['class_name']}' -> {cocok}")
ids = set(pairs["class_id"].tolist())
print(f"DATA AKTUAL PROJECT: class_id di file = {sorted(ids)}; ekspektasi = {sorted(EXPECTED)}")
print(f"DATA AKTUAL PROJECT: kolom provenance tersedia = {[c for c in ['source', 'source_class'] if c in gpd.read_file(TRAIN_GEOJSON, rows=1).columns]} (dirinci di Bab 2)")

DATA AKTUAL PROJECT: pasangan unik class_id/class_name di file = 6
  class_id=1 class_name='Sawah' -> COCOK
  class_id=2 class_name='Bangunan / Permukiman' -> COCOK
  class_id=3 class_name='Mangrove' -> COCOK
  class_id=4 class_name='Lahan hijau' -> COCOK
  class_id=5 class_name='Laut' -> COCOK
  class_id=6 class_name='Danau' -> COCOK
DATA AKTUAL PROJECT: class_id di file = [1, 2, 3, 4, 5, 6]; ekspektasi = [1, 2, 3, 4, 5, 6]
DATA AKTUAL PROJECT: kolom provenance tersedia = ['source', 'source_class'] (dirinci di Bab 2)


**Interpretasi:** pasangan unik `class_id`/`class_name` di file dicocokkan dengan tabel 1.4. Bila semua berstatus COCOK dan himpunan id sama dengan ekspektasi, maka definisi 6 kelas pada notebook ini konsisten dengan isi file. Bila ada TIDAK COCOK, maka itu temuan Bab 2 (kriteria V4) dan dilaporkan beserta jumlahnya. Rincian kolom provenance (`source`, `source_class`) ditabulasikan di Bab 2.

## 1.5 Acuan klasifikasi lahan Indonesia

### Acuan 1 — SNI 7645 (klasifikasi penutup lahan, BSN)

- **Dokumen terverifikasi:** SNI 7645:2010 *Klasifikasi penutup lahan* (BSN; berkas PDF diunduh dari portal BIG), direvisi menjadi **SNI 7645-1:2014** *Klasifikasi penutup lahan — Bagian 1: Skala kecil dan menengah* (Komite Teknis 07-01 Informasi Geografi/Geomatika; status Berlaku; dikonfirmasi 2020 tanpa perubahan substansi). Hierarki skala: 1:1.000.000 (kecil), 1:250.000 / 1:50.000 / 1:25.000 (menengah). Disusun mengacu **LCCS UN-FAO** dan **ISO 19144-1**.
- **Sumber:** 
    - (1) https://big.go.id/assets/download/sni/SNI/15.%20SNI%207645-2010%20Klasifikasi%20penutup%20lahan.pdf.
    - (2) https://pesta.bsn.go.id/produk/detail/10269-sni7645-12014 (halaman produk BSN SNI 7645-1:2014).
    - (3) salinan PDF SNI 7645-1:2014 (Konfirmasi 2020) via BBPSIK-Jogja https://jogja.bsilhk.menlhk.go.id/wp-content/uploads/2023/09/5.-SNI-7645-1_2014-_Klasifikasi-penutup-lahan-bag-1-skala-kecil-dan-menengah.pdf (diakses 2026-10-07).
- **Struktur umum terverifikasi:** dua cabang besar — *area dominan bukan-vegetasi* (tubuh air, lahan terbuka, bangunan) dan *area dominan vegetasi* (hutan, perkebunan, sawah, semak, herba/rumput). Skala 1:50.000/1:25.000 mulai memasukkan unsur penggunaan lahan (*land use*).

### Acuan 2 — Klasifikasi penutup lahan KLHK (Direktorat IPSDH / Ditjen Planologi Kehutanan)

- **Dokumen terverifikasi:** Juknis Penafsiran Citra Satelit Resolusi Menengah untuk Pemutakhiran Penutupan Lahan Nasional (SK Dirjen Planologi Kehutanan No. 17), **Lampiran 3: Sistem Klasifikasi dan Monogram Data Penutupan Lahan** — memuat **23 kelas** penutupan lahan nasional, diproduksi sejak 1990 lewat interpretasi visual citra resolusi menengah.
- **Sumber:** https://sigap.kehutanan.go.id/sigap-admin-2026/files/download/sk-dirjen-planologi-kehutanan-no-17-tentang-juknis-penafsiran-csrm-untuk-pemutakhiran-pl-nasional.pdf ; penjelasan metode: https://www.indonesia-geospasial.com/2021/04/petunjuk-pembuatan-shapefile-penutupan.html (memuat urutan logika prioritas: primer = penutupan/biofisik, sekunder = penggunaan/aktivitas manusia). Diakses 2026-10-07.
- **Daftar nama 23 kelas terverifikasi dari Lampiran 3:** Hutan Lahan Kering Primer; Hutan Lahan Kering Sekunder; Hutan Mangrove Primer; Hutan Mangrove Sekunder; Hutan Rawa Primer; Hutan Rawa Sekunder; Hutan Tanaman; Semak Belukar; Semak Belukar Rawa; Pertanian Lahan Kering; Pertanian Lahan Kering Campur; Sawah; Tambak; Transmigrasi; Perkebunan; Permukiman; Bandara/Pelabuhan; Lahan Terbuka; Pertambangan; Tubuh Air; Rawa; Savana/Padang Rumput; Awan. **Kode numerik tiap kelas: tidak terverifikasi** (fragmen kode yang beredar tidak memuat pemetaan lengkap yang dapat dikutip andal), sehingga tidak ditulis di sini.

### Pemetaan 6 kelas 

| Kelas project | Padanan SNI 7645-1:2014 (nama + kode bila terverifikasi) | Padanan KLHK 23 kelas (nama; kode: tidak terverifikasi) | Status |
|---|---|---|---|
| 1 Sawah | Tanaman semusim lahan basah (sawah): 2.2.1.6.1 Sawah dengan padi terus menerus; 2.2.1.6.2 Sawah dengan padi diselingi tanaman lain/bera | Sawah | Operasional ≈ acuan (dinamika bera/panen pada snapshot dibahas di Bab 2) |
| 2 Bangunan / Permukiman | 1.2.3.1.1 Bangunan permukiman kota; 1.2.3.1.2 Bangunan permukiman desa (berasosiasi vegetasi pekarangan) | Permukiman (+ Transmigrasi bila relevan) | Operasional ≈ acuan |
| 3 Mangrove | Hutan mangrove primer/sekunder per kerapatan (kode rinci: tidak terverifikasi — penomoran berbeda antar edisi 2010 vs 2014) | Hutan Mangrove Primer / Hutan Mangrove Sekunder | Operasional ≈ acuan |
| 4 Lahan hijau | AGREGAT dari banyak kelas SNI: hutan (jati, pinus, dsb.), perkebunan, kebun campuran (2.2.1.4.x), ladang/tegalan, semak belukar (2.1.1.7.x), padang rumput/alang-alang (2.1.1.8.x) | AGREGAT: Hutan Lahan Kering (Primer/Sekunder), Hutan Tanaman, Perkebunan, Pertanian Lahan Kering (Campur), Semak Belukar, Savana/Padang Rumput | Agregasi operasional — BUKAN kelas resmi tunggal |
| 5 Laut | 1.1.1.1 Perairan laut (1.1.1.1.1 dangkal / 1.1.1.1.2 dalam) | Tubuh Air (bagian laut) | Operasional ≈ acuan (lengkapnya di Bab 2: buffer pantai NE 10m) |
| 6 Danau | 1.1.1.2 Danau/telaga alami; 1.2.1.1 Waduk dan danau buatan (1.2.1.1.1–1.2.1.1.5) | Tubuh Air (bagian danau/waduk) | Operasional ≈ acuan (termasuk waduk/embung buatan) |

**Apa yang TIDAK tertangkap 6 kelas ini (keterbatasan desain):** pertanian lahan kering/tegalan non-sawah, tambak (asin/payau), hutan primer vs sekunder vs tanaman, semak vs belukar vs alang-alang, lahan terbuka/pertambangan, sungai/rawa, awan/bayangan awan — semuanya tercampur ke dalam agregat (terutama "Lahan hijau") atau tidak dimodelkan. Kelas awan tidak ada sehingga sisa awan pada citra (diperiksa di Bab 2) cenderung menjadi sumber kesalahan yang harus dilaporkan.

## 1.6 Apa yang diprediksi model

| Istilah | File project | Isi / peran |
|---|---|---|
| **Raster input** | `data/s2_jatim.tif` (+ `data/s2_jatim_metadata.json`) | Citra Sentinel-2 komposit (band B02, B03, B04, B08, B11, B12). Setiap pixel menyimpan nilai reflektansi per band. Dibaca/diproses **per window/blok** (raster besar). |
| **Training data** | `data/training_samples_jawa_timur.geojson` / `.gpkg` → (Bab 3) `outputs/tables/training_pixel.csv`, `training_mean.csv`, `polygon_split.csv` | Polygon berlabel (maks 50 valid per kelas) yang diekstraksi menjadi fitur ML. Split **per polygon (FID)**, tuning via GroupKFold pada training saja. |
| **Testing data** | (Bab 3) `outputs/tables/testing_pixel.csv`, `testing_mean.csv` | Polygon terpilih yang ditahan; dipakai **sekali di akhir** (Bab 5). Bukan dasar pemilihan model. |
| **Classification result** | `outputs/raster/klasifikasi_jawa_timur.tif` | Peta hasil: tiap pixel menyimpan **`class_id` prediksi (1–6)** dari model terbaik. Ditampilkan di WebGIS (notebook 06) + Streamlit. |

**Yang sebenarnya diprediksi model:** untuk tiap pixel, model menerima vektor fitur (nilai band / indeks spektral hasil ekstraksi Bab 3) dan mengeluarkan satu `class_id` (1–6). CONTOH ILUSTRASI — bukan data project: mis. pixel dengan NIR tinggi dan SWIR rendah *cenderung* diarahkan model ke kelas vegetasi — ilustrasi cara membaca fitur, bukan aturan keputusan dan bukan angka project.

## 1.7 Alur target akhir

```text
Sentinel-2 (s2_jatim.tif + metadata)
        +
Training polygon (training_samples_jawa_timur.*)
        |
        v
Ekstraksi fitur per polygon (Bab 3: pixel-based + polygon-mean)
        |
        v
ML + pemilihan via CV GroupKFold pada training (Bab 4)
        |
        v
Evaluasi sekali pada test + luas per kelas (Bab 5)
        |
        v
Model terbaik -> klasifikasi seluruh raster per window -> klasifikasi_jawa_timur.tif
        |
        +----------------------+----------------------+
        v                      v
WebGIS di notebook 06        Streamlit app.py (pelengkap)
```

Pemilihan model berdasarkan CV training; test hanya konfirmasi akhir. Detail tiap panah dikerjakan di Bab 2–6.

## Lampiran A — Bukti file project (DATA AKTUAL PROJECT)

Seluruh angka di bawah dicetak dari variabel hasil perhitungan (f-string)..

In [2]:
# DATA AKTUAL PROJECT — A1 inventaris file input (ukuran dibaca dari disk).
import os

DATA_DIR = "../data"
FILES = ["boundary_jatim.geojson", "s2_jatim.tif", "s2_jatim_metadata.json",
         "training_samples_jawa_timur.geojson", "training_samples_jawa_timur.gpkg"]
print("DATA AKTUAL PROJECT: inventaris 5 file input (0.3)")
for fn in FILES:
    p = os.path.join(DATA_DIR, fn)
    ada = os.path.exists(p)
    ukuran = os.path.getsize(p) if ada else -1
    print(f"  {fn}: ada={ada}, ukuran_byte={ukuran}")
print("DATA AKTUAL PROJECT: geoboundaries_IDN_ADM1.geojson -> TIDAK TERPAKAI (tidak dibaca)")

DATA AKTUAL PROJECT: inventaris 5 file input (0.3)
  boundary_jatim.geojson: ada=True, ukuran_byte=151929
  s2_jatim.tif: ada=True, ukuran_byte=1564425175
  s2_jatim_metadata.json: ada=True, ukuran_byte=543
  training_samples_jawa_timur.geojson: ada=True, ukuran_byte=11045852
  training_samples_jawa_timur.gpkg: ada=True, ukuran_byte=4788224
DATA AKTUAL PROJECT: geoboundaries_IDN_ADM1.geojson -> TIDAK TERPAKAI (tidak dibaca)


**Interpretasi A1:** kelima file input harus berstatus `ada=True`. Ukuran byte sekadar bukti keberadaan (raster terbesar). Bila ada yang `False`, pipeline berhenti — Bab berikutnya tidak dapat dikerjakan tanpa mengarang data.

In [3]:
# DATA AKTUAL PROJECT — A2 isi aktual metadata openEO vs pernyataan prompt 0.5.
import json

with open("../data/s2_jatim_metadata.json", encoding="utf-8") as f:
    meta = json.load(f)
print("DATA AKTUAL PROJECT: isi s2_jatim_metadata.json =")
print(json.dumps(meta, indent=2, ensure_ascii=False))
print()
# Perbandingan dengan blok 0.5 (nilai ekspektasi ditulis sebagai pembanding, bukan hasil):
print(f"DATA AKTUAL PROJECT: koleksi tercatat = {meta.get('koleksi', 'tidak tercatat di metadata')}")
print(f"DATA AKTUAL PROJECT: bands tercatat = {meta.get('bands', 'tidak tercatat di metadata')}")
print(f"DATA AKTUAL PROJECT: temporal_extent tercatat = {meta.get('temporal_extent', 'tidak tercatat di metadata')}")
print(f"DATA AKTUAL PROJECT: komposit/reducer tercatat = {meta.get('komposit', 'tidak tercatat di metadata')}")
print(f"DATA AKTUAL PROJECT: max_cloud_cover_pct tercatat = {meta.get('max_cloud_cover_pct', 'tidak tercatat di metadata')}")
print(f"DATA AKTUAL PROJECT: bbox tercatat = {meta.get('bbox', 'tidak tercatat di metadata')}")
for k in ["scale_factor_reflectance", "resampling", "cloud_mask", "collection_L1C_L2A_detail"]:
    print(f"DATA AKTUAL PROJECT: {k} = {meta.get(k, 'tidak tercatat di metadata')}")

DATA AKTUAL PROJECT: isi s2_jatim_metadata.json =
{
  "koleksi": "SENTINEL2_L2A",
  "tanggal_mulai": "2025-09-01",
  "tanggal_akhir": "2025-09-02",
  "temporal_extent": [
    "2025-09-01",
    "2025-09-03"
  ],
  "komposit": "median",
  "bands": [
    "B02",
    "B03",
    "B04",
    "B08",
    "B11",
    "B12"
  ],
  "max_cloud_cover_pct": 20,
  "bbox": {
    "west": 110.8988,
    "south": -8.7804,
    "east": 116.2701,
    "north": -5.0489,
    "crs": "EPSG:4326"
  },
  "aoi_file": "boundary_jatim.geojson",
  "output": "s2_jatim.tif",
  "job_id": "j-26100607452343a2be0930a04fc04519"
}

DATA AKTUAL PROJECT: koleksi tercatat = SENTINEL2_L2A
DATA AKTUAL PROJECT: bands tercatat = ['B02', 'B03', 'B04', 'B08', 'B11', 'B12']
DATA AKTUAL PROJECT: temporal_extent tercatat = ['2025-09-01', '2025-09-03']
DATA AKTUAL PROJECT: komposit/reducer tercatat = median
DATA AKTUAL PROJECT: max_cloud_cover_pct tercatat = 20
DATA AKTUAL PROJECT: bbox tercatat = {'west': 110.8988, 'south': -8.7804, 'east': 

**Interpretasi A2:** bandingkan isi aktual dengan blok 0.5 (`tanggal_mulai 2025-09-01`, `tanggal_akhir 2025-09-02`, `temporal_extent [2025-09-01, 2025-09-03]`). Bila sama, nyatakan sama; bila beda, tampilkan keduanya dan jelaskan perbedaannya dari isi file. `koleksi` menunjukkan L1C/L2A bila tercatat. Batas akhir `temporal_extent` pada konvensi openEO umumnya eksklusif (perlu verifikasi dari perilaku, bukan asumsi) — implikasinya dibahas di Bab 2 bersama pemeriksaan celah/nodata, sisa awan, dan garis mosaik (rentang sangat pendek sehingga risiko tersebut wajib diperiksa dari data). Yang tidak tercatat ditulis "tidak tercatat di metadata".

In [ ]:
# DATA AKTUAL PROJECT — A3 bukti boundary dari ISI geometri
import json
from shapely.geometry import shape

with open("../data/boundary_jatim.geojson", encoding="utf-8") as f:
    bj = json.load(f)
feats = bj.get("features", [])
geoms = [shape(ft["geometry"]) for ft in feats if ft.get("geometry")]
bounds = [g.bounds for g in geoms]  # (minx, miny, maxx, maxy) = (lon min, lat min, lon maks, lat maks)
minx = min(b[0] for b in bounds)
miny = min(b[1] for b in bounds)
maxx = max(b[2] for b in bounds)
maxy = max(b[3] for b in bounds)
print(f"DATA AKTUAL PROJECT: tipe koleksi = {bj.get('type')}, n_features = {len(feats)}")
print(f"DATA AKTUAL PROJECT: tipe geometri = {sorted(set(f['geometry']['type'] for f in feats))}")
print(f"DATA AKTUAL PROJECT: bbox geometri lon [{minx:.4f}, {maxx:.4f}], lat [{miny:.4f}, {maxy:.4f}]")
print(f"DATA AKTUAL PROJECT: dalam rentang Jawa Timur (lon ~111-115, lat ~-8.8 s.d. -5)? "
      f"{110.0 <= minx <= 112.0 and 114.0 <= maxx <= 117.0 and -9.5 <= miny <= -7.0 and -6.0 <= maxy <= -4.5}")

DATA AKTUAL PROJECT: tipe koleksi = FeatureCollection, n_features = 1
DATA AKTUAL PROJECT: tipe geometri = ['MultiPolygon']
DATA AKTUAL PROJECT: bbox geometri lon [110.8988, 116.2701], lat [-8.7804, -5.0489]
DATA AKTUAL PROJECT: dalam rentang Jawa Timur (lon ~111-115, lat ~-8.8 s.d. -5)? True


**Interpretasi A3:** bila bbox geometri berada pada rentang bujur/lintang Jawa Timur dan terdiri dari satu/lebih poligon darat-plus-pesisir, maka file tersebut terbukti memuat boundary Jawa Timur dari isinya. Sifat detail (apakah mencakup area laut 25 km, dsb.) diperiksa terhadap raster di Bab 2.

In [5]:
# DATA AKTUAL PROJECT — A4 profil raster (header saja; raster besar TIDAK dimuat sekaligus).
import rasterio

with rasterio.open("../data/s2_jatim.tif") as src:
    print(f"DATA AKTUAL PROJECT: count = {src.count}, dtype = {src.dtypes[0]}, crs = {src.crs}")
    print(f"DATA AKTUAL PROJECT: width = {src.width}, height = {src.height}")
    print(f"DATA AKTUAL PROJECT: bounds = {src.bounds}")
    print(f"DATA AKTUAL PROJECT: band names/descriptions = {[src.descriptions[i] for i in range(src.count)]}")
    print(f"DATA AKTUAL PROJECT: nodata = {src.nodata}")

DATA AKTUAL PROJECT: count = 6, dtype = int16, crs = EPSG:32749
DATA AKTUAL PROJECT: width = 59412, height = 41323
DATA AKTUAL PROJECT: bounds = BoundingBox(left=488850.0, bottom=9027590.0, right=1082970.0, top=9440820.0)
DATA AKTUAL PROJECT: band names/descriptions = ['B02', 'B03', 'B04', 'B08', 'B11', 'B12']
DATA AKTUAL PROJECT: nodata = -32768.0


**Interpretasi A4:** jumlah band pada header dicocokkan dengan enam band metadata (B02, B03, B04, B08, B11, B12). CRS, ukuran, dan bounds menunjukkan cakupan raster aktual. Nilai pixel, statistik, dan pemeriksaan nodata/awan/mosaik dikerjakan per window di Bab 2 — tidak di sini.

In [6]:
# DATA AKTUAL PROJECT — A5 kolom aktual file training (nama kolom diperiksa dulu, tidak diasumsikan).
import geopandas as gpd

g1 = gpd.read_file("../data/training_samples_jawa_timur.geojson", rows=5)
print(f"DATA AKTUAL PROJECT: kolom geojson = {list(g1.columns)}")
g2 = gpd.read_file("../data/training_samples_jawa_timur.gpkg", rows=5)
print(f"DATA AKTUAL PROJECT: kolom gpkg = {list(g2.columns)}")
print(f"DATA AKTUAL PROJECT: kolom geometri geojson = {g1.geometry.geom_type.unique().tolist()}")

DATA AKTUAL PROJECT: kolom geojson = ['fid', 'class_id', 'class_name', 'source', 'source_class', 'geometry']
DATA AKTUAL PROJECT: kolom gpkg = ['class_id', 'class_name', 'source', 'source_class', 'geometry']
DATA AKTUAL PROJECT: kolom geometri geojson = ['Polygon']


**Interpretasi A5:** nama kolom aktual menentukan kode Bab 3 (tidak mengasumsikan nama kolom). Kolom provenance (`source`, `source_class`) dan tipe geometri yang terlihat di sini dirinci dan disaring di Bab 2 (kriteria V1–V6). File `.geojson` dan `.gpkg` diharapkan memuat informasi yang setara; bila berbeda, Bab 2 mencatat perbedaannya dan memilih satu sebagai acuan.
 Terkonfirmasi di Bab 2: kedua file berisi 2823 fitur dengan komposisi atribut identik; kolom `fid` hanya ada di GeoJSON.